In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
data = pd.read_csv(url)

# Data Exploration
print(data.info())
print(data.describe())
print(data['Survived'].value_counts(normalize=True))

# Visualize survival rates by sex and class
plt.figure(figsize=(8, 6))
sns.barplot(x='Pclass', y='Survived', hue='Sex', data=data)
plt.title('Survival Rates by Sex and Passenger Class')
plt.savefig('survival_plot.png')
plt.close()

# Data Cleaning
# Impute missing Age with median by Pclass
data['Age'] = data.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
# Impute missing Embarked with mode
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)
# Drop irrelevant columns
data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# Feature Engineering
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
# Cap Fare outliers
data['Fare'] = data['Fare'].clip(upper=data['Fare'].quantile(0.95))
# Encode categorical variables
le = LabelEncoder()
data['Sex'] = le.fit_transform(data['Sex'])
data = pd.get_dummies(data, columns=['Pclass', 'Embarked'], drop_first=True)

# Prepare data for modeling
X = data.drop('Survived', axis=1)
y = data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}
metrics = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    })

# Display metrics
metrics_df = pd.DataFrame(metrics)
print(metrics_df)

# Feature importance for Random Forest
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
importances.sort_values().plot(kind='barh')
plt.title('Feature Importance (Random Forest)')
plt.savefig('feature_importance.png')
plt.close()
